# Data Download

In [1]:
import sys
import urllib.request
import zipfile
from pathlib import Path

In [5]:
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
try:
    _notebook_dir = Path(__file__).resolve().parent
except NameError:
    _notebook_dir = Path().resolve()  # Jupyter: cwd is the notebook's directory

RAW_DIR = _notebook_dir.parent / "data" / "raw"
DEST_DIR = RAW_DIR / "movielens"
ZIP_PATH = RAW_DIR / "ml-100k.zip"
REQUIRED_FILES = ["u.data", "u.item"]

In [2]:
def _progress_hook(block_num: int, block_size: int, total_size: int) -> None:
    downloaded = block_num * block_size
    if total_size > 0:
        pct = min(downloaded * 100 / total_size, 100)
        print(f"\r  {pct:5.1f}%  ({downloaded // 1024} / {total_size // 1024} KB)", end="", flush=True)

In [6]:
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
urllib.request.urlretrieve(MOVIELENS_URL, ZIP_PATH, reporthook=_progress_hook)

  100.0%  (4816 / 4808 KB)

(WindowsPath('Z:/Personal project/recommendation_system/data/raw/ml-100k.zip'),
 <http.client.HTTPMessage at 0x1bffdc66d50>)

In [7]:
DEST_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    # ml-100k.zip contains a top-level ml-100k/ folder; strip it
    for member in zf.infolist():
        # member.filename looks like "ml-100k/u.data"
        parts = Path(member.filename).parts
        if len(parts) < 2:
            continue
        relative = Path(*parts[1:])
        target = DEST_DIR / relative
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(zf.read(member.filename))

In [8]:
missing = [f for f in REQUIRED_FILES if not (DEST_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing files after extraction: {missing}")
for name in REQUIRED_FILES:
    size = (DEST_DIR / name).stat().st_size